# Tugas 1 - Klasifikasi Jenis Suara dengan kNN

Dataset `voice.csv` berisi 3.168 data suara dengan 20 fitur akustik dan 1 label (`male`/`female`). Tujuan tugas ini adalah membangun model kNN untuk mengklasifikasikan jenis suara, memilih fitur yang paling optimal, serta menentukan nilai K terbaik.

## 1. Load Data

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Cari dataset: folder lokal -> /content (Colab) -> Google Drive -> GitHub
GITHUB_URL = 'https://raw.githubusercontent.com/oninonins/Machine-learning-/main/JS09/voice.csv'
DATA_CANDIDATES = [
    'voice.csv',
    '/content/voice.csv',
    '/content/drive/MyDrive/voice.csv',
    GITHUB_URL,
]

DATA_PATH = next(
    (p for p in DATA_CANDIDATES
     if not p.startswith('http') and os.path.exists(p)),
    None,
)

if DATA_PATH is None:
    DATA_PATH = GITHUB_URL

try:
    data = pd.read_csv(DATA_PATH)
except Exception as exc:
    raise FileNotFoundError(
        'voice.csv tidak ditemukan. Letakkan file di folder notebook, '
        'upload ke /content, atau sesuaikan DATA_CANDIDATES.'
    ) from exc

print('Dataset:', DATA_PATH)
data.head()

## 2. Eksplorasi Data

In [ ]:
# ukuran dan struktur data
print('Ukuran data:', data.shape)
data.info()

In [ ]:
# statistik deskriptif
data.describe()

In [ ]:
# cek missing value, duplikat, dan distribusi label
print('Jumlah missing value:', data.isnull().sum().sum())
print('Jumlah data duplikat:', data.duplicated().sum())
print(data['label'].value_counts())

## 3. Preprocessing

Dua baris duplikat dihapus agar data yang sama tidak muncul sekaligus pada data latih dan data uji. Label diubah menjadi angka (`male = 0`, `female = 1`).

In [ ]:
# hapus data duplikat
data = data.drop_duplicates().reset_index(drop=True)
print('Ukuran data setelah drop duplikat:', data.shape)

# encode label: male = 0, female = 1
data['label'] = data['label'].map({'male': 0, 'female': 1})
data['label'].value_counts()

## 4. Visualisasi Data

In [ ]:
# distribusi label dan contoh fitur dengan korelasi tertinggi
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=data, x='label', ax=axes[0])
axes[0].set_title('Distribusi Kelas Suara')
axes[0].set_xlabel('label (0 = male, 1 = female)')

sns.boxplot(data=data, x='label', y='meanfun', ax=axes[1])
axes[1].set_title('meanfun berdasarkan Kelas')
axes[1].set_xlabel('label (0 = male, 1 = female)')

plt.tight_layout()
plt.show()

In [ ]:
# heatmap korelasi seluruh fitur
plt.figure(figsize=(14, 10))
sns.heatmap(data.corr(), cmap='YlGnBu', center=0)
plt.title('Heatmap Korelasi Fitur voice.csv')
plt.show()

## 5. Pemisahan Data dan Standardisasi

Data dibagi 70% latih dan 30% uji dengan stratifikasi agar proporsi kelas tetap seimbang. Standardisasi dilakukan karena kNN berbasis jarak.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = data.drop('label', axis=1)
y = data['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print('Ukuran X_train:', X_train.shape)
print('Ukuran X_test :', X_test.shape)

## 6. Model Baseline kNN

Sebagai pembanding, model kNN dilatih menggunakan seluruh 20 fitur dengan `k = 5`.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

knn_base = KNeighborsClassifier(n_neighbors=5)
knn_base.fit(X_train_sc, y_train)

y_pred_base = knn_base.predict(X_test_sc)
print('Akurasi baseline (semua fitur, k=5):', accuracy_score(y_test, y_pred_base))

## 7. Analisis Fitur dengan Mutual Information

Mutual Information digunakan untuk mengukur seberapa besar informasi setiap fitur terhadap label. Semakin tinggi skornya, semakin informatif fitur tersebut.

In [ ]:
from sklearn.feature_selection import SelectKBest, mutual_info_classif

def mi_score(X, y):
    return mutual_info_classif(X, y, random_state=42)

selector = SelectKBest(score_func=mi_score, k='all')
selector.fit(X_train_sc, y_train)

skor_mi = pd.DataFrame({
    'Fitur': X.columns,
    'MI Score': selector.scores_
}).sort_values('MI Score', ascending=False).reset_index(drop=True)

skor_mi

In [ ]:
# visualisasi peringkat fitur
plt.figure(figsize=(10, 6))
sns.barplot(data=skor_mi, x='MI Score', y='Fitur', color='steelblue')
plt.title('Peringkat Fitur berdasarkan Mutual Information')
plt.xlabel('MI Score')
plt.ylabel('Fitur')
plt.show()

## 8. Pencarian Jumlah Fitur dan Nilai K Terbaik

Pipeline `StandardScaler -> SelectKBest -> KNeighborsClassifier` diuji dengan `GridSearchCV` 5-fold. Kombinasi yang dicari adalah jumlah fitur 1-20 dan nilai K ganjil 1-29.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('selector', SelectKBest(score_func=mi_score)),
    ('knn', KNeighborsClassifier())
])

param_grid = {
    'selector__k': list(range(1, 21)),
    'knn__n_neighbors': list(range(1, 30, 2))
}

grid = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X_train, y_train)

print('Parameter terbaik:', grid.best_params_)
print('Akurasi CV terbaik:', grid.best_score_)

## 9. Evaluasi Model Terbaik

In [ ]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print('Akurasi data uji:', accuracy_score(y_test, y_pred))
print('\nConfusion Matrix:\n', confusion_matrix(y_test, y_pred))
print('\nClassification Report:\n', classification_report(
    y_test, y_pred, target_names=['male', 'female']
))

In [ ]:
# tampilkan fitur yang terpilih
selected_features = list(X.columns[best_model.named_steps['selector'].get_support()])
print('Fitur terpilih:', selected_features)

## 10. Grafik Analisis Jumlah Fitur dan Nilai K

In [ ]:
from sklearn.model_selection import cross_val_score

best_k = grid.best_params_['knn__n_neighbors']
n_features = list(range(1, 21))
cv_scores = []

for n in n_features:
    model = Pipeline([
        ('scaler', StandardScaler()),
        ('selector', SelectKBest(score_func=mi_score, k=n)),
        ('knn', KNeighborsClassifier(n_neighbors=best_k))
    ])
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1)
    cv_scores.append(scores.mean())

plt.figure(figsize=(8, 5))
plt.plot(n_features, cv_scores, marker='o')
plt.axvline(x=grid.best_params_['selector__k'], color='r', linestyle='--',
            label='Jumlah fitur terbaik')
plt.title('Jumlah Fitur vs Akurasi Cross-Validation')
plt.xlabel('Jumlah fitur')
plt.ylabel('Akurasi CV')
plt.xticks(n_features)
plt.legend()
plt.show()

In [ ]:
# kurva akurasi train dan test terhadap nilai K
ks = list(range(1, 30, 2))
acc_train = []
acc_test = []

for k in ks:
    model = Pipeline([
        ('scaler', StandardScaler()),
        ('selector', SelectKBest(score_func=mi_score, k=grid.best_params_['selector__k'])),
        ('knn', KNeighborsClassifier(n_neighbors=k))
    ])
    model.fit(X_train, y_train)
    acc_train.append(model.score(X_train, y_train))
    acc_test.append(model.score(X_test, y_test))

plt.figure(figsize=(8, 5))
plt.plot(ks, acc_train, marker='o', label='Train')
plt.plot(ks, acc_test, marker='o', label='Test')
plt.axvline(x=best_k, color='r', linestyle='--', label='K terbaik')
plt.title('Nilai K vs Akurasi')
plt.xlabel('Nilai K')
plt.ylabel('Akurasi')
plt.legend()
plt.show()

## 11. Analisis Hasil

### Fitur yang digunakan

Berdasarkan `SelectKBest` dengan Mutual Information dan `GridSearchCV`, jumlah fitur optimal adalah **7 fitur**, yaitu:

| Peringkat | Fitur | MI Score |
|---|---|---|
| 1 | `meanfun` | 0.5591 |
| 2 | `IQR` | 0.4528 |
| 3 | `Q25` | 0.4062 |
| 4 | `sd` | 0.3037 |
| 5 | `mode` | 0.1792 |
| 6 | `sp.ent` | 0.1654 |
| 7 | `sfm` | 0.1351 |

`meanfun` (rata-rata frekuensi fundamental) merupakan fitur paling informatif karena perbedaan pitch suara male dan female sangat jelas. Fitur distribusi frekuensi seperti `IQR`, `Q25`, dan `sd` juga membantu memisahkan kedua kelas.

### Nilai K terbaik

Nilai **K = 3** menghasilkan akurasi terbaik dengan **akurasi cross-validation 0.9801** dan **akurasi data uji 0.9821**.

- **K = 1** memberikan akurasi latih sempurna (1.0000), tetapi akurasi uji lebih rendah. Kondisi ini menunjukkan overfitting karena model hanya mengikuti satu tetangga terdekat yang sangat sensitif terhadap noise.
- **K = 3** memberikan keseimbangan terbaik antara akurasi latih (0.9878) dan akurasi uji (0.9821).
- Nilai K yang semakin besar cenderung menurunkan akurasi uji karena batas keputusan menjadi terlalu halus (underfitting).

### Perbandingan dengan baseline

Model baseline dengan seluruh 20 fitur dan `k = 5` memperoleh akurasi **0.9779**. Setelah fitur diseleksi menjadi 7 fitur dan `k = 3`, akurasi meningkat menjadi **0.9821**. Penambahan fitur setelah 7 fitur justru menurunkan akurasi cross-validation (misalnya 20 fitur menjadi 0.9702), sehingga fitur yang kurang informatif cenderung menambah noise dan dimensi data.

### Evaluasi klasifikasi

Confusion matrix model terbaik:

```
[[465  10]
 [  7 468]]
```

Model mampu mengklasifikasikan kelas **male** dan **female** dengan precision, recall, dan F1-score sekitar **0.98** untuk kedua kelas. Dengan demikian, kombinasi **7 fitur terbaik dan K = 3** sudah sangat baik untuk kasus klasifikasi jenis suara ini.